# Machine Learning for Statistical NLP: Advanced
# Assignment 01

## Part 1 - Tokenizers

In [14]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Python version:", sys.version)

Python: /home/gusbeihel@GU.GU.SE/MaLearn_Adv/MaLearn_Adv_Assignment01/.venv/bin/python
Python version: 3.12.12 (main, Mar 27 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-11)]


Checking Assignment Data

In [15]:
DATA_DIR = Path("/srv/data/lt2326-h26/a1")

print("Data directory exists:", DATA_DIR.exists())

for split in ["train", "valid", "test"]:
    for lang in ["en", "tr", "zh"]:
        path = DATA_DIR / split / f"{lang}.txt"
        print(f"{split:5} {lang}: {path.exists()}")

Data directory exists: True
train en: True
train tr: True
train zh: True
valid en: True
valid tr: True
valid zh: True
test  en: True
test  tr: True
test  zh: True


Checking tokenizer training files + metadata

In [16]:
for filename in [
    "metadata.json",
    "README.txt",
    "tokenizer/balanced.txt",
    "tokenizer/imbalanced_10-1-1.txt",
]:
    path = DATA_DIR / filename
    print(f"{filename}: {path.exists()}")

metadata.json: True
README.txt: True
tokenizer/balanced.txt: True
tokenizer/imbalanced_10-1-1.txt: True


Checking necessary packages

In [17]:
import pandas as pd
import matplotlib
import tokenizers

print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("tokenizers:", tokenizers.__version__)

pandas: 3.0.6
matplotlib: 3.11.2
tokenizers: 0.23.2


### Data Inspection

In [18]:
# README.text
file_1 = open("/srv/data/lt2326-h26/a1/README.txt", "r")
content = file_1.read()
print(content)
file_1.close()

# metadata.json
file_2 = open("/srv/data/lt2326-h26/a1/metadata.json", "r")
content_2 = file_2.read()
print(content_2)
file_2.close()

Multilingual tokenization assignment data

Languages: English (en), Turkish (tr), Mandarin Chinese (zh)
Source: Leipzig Corpora Collection, Wikipedia sentence corpora, 2021, 100K

Splits
------
train/en.txt, train/tr.txt, train/zh.txt
valid/en.txt, valid/tr.txt, valid/zh.txt
test/en.txt,  test/tr.txt,  test/zh.txt

Tokenizer-training corpora
--------------------------
tokenizer/balanced.txt
    Combined training sentences with approximately equal underlying character counts per language.

tokenizer/imbalanced_10-1-1.txt
    Same training pool, but approximately EN:TR:ZH = 10:1:1 by Unicode character count.
    Intended for the optional imbalance experiment. Do not use it as LM evaluation data.

Important design choice
-----------------------
The languages are balanced by Unicode character count rather than sentence count. This is not a
claim that one Unicode character carries equal linguistic information across languages; it is simply
a transparent, tokenizer-independent unit for const

# !!!
Data notes:
- NFC Unicode normalization
- whitespace collapsed
- empty sentences removed
- exact duplicates removed within languages
- fixed-seed shuffle
- languages balanced by Unicode code-point count

### Predictions
- Which language do you expect to require the most tokens?   
For the character tokenizer, each language should have roughly similar token counts (since the characters are balanced in the data).  
For BPE, I expect Chinese, because it's a character based language, and one character can already be a word. So two different characters might not occur with each other that much, therefore there might be more single character tokens. It might also not be as morphologically rich as Turkish, or even English in comparison.

- How do you expect the larger BPE vocabulary to change the representation of each language?  
I think that each language will have less tokens in total and per sentence and the number of characters per token will go up (since the larger vocabulary will allow longer sequences to become tokens). I expect that Turkish especially will receive noticeably less tokens since it might originally have had more tokens because it's morphologically rich. I expect a similar outcome for English, but less so. For Chinese, the number of characters per token might be lower than for English and Turkish due to it being a character-based language. But it might lead to tokens being expressions or even short utterances.   

- What kinds of units do you expect BPE to learn in English, Turkish, and Chinese?  
For English, I mostly expect BPE to learn tokens that are words, and maybe some suffixes. For Turkish - a morphologically richer language -, I expect it to learn more affixes and stems, as well as words. For Chinese, I expect it to mostly learn characters, possibly words made up of several characters or even expressions made up of several characters.  

- Which tokenizer do you expect to work best for language modeling?  
I think the small BPE tokenizer might be the sweet spot. It will probably create larger meaningful chunks while not putting rare larger tokens at a disadvantage in training by creating too rare tokens.


### Character-Level Tokenizer

In [19]:
## building the vocabulary

balanced_path = DATA_DIR / "tokenizer" / "balanced.txt"

# inspecting the data
balanced_train = open(balanced_path, encoding="utf-8")
line = balanced_train.readline()

print(line)
print(type(line))
print(len(line))

#for char in line[:20]:
    #print(repr(char))

balanced_train.close()

# building a character set
characters = set()

with open(balanced_path, encoding="utf-8") as balanced_train: # reopening to start from the first line again

    for line in balanced_train:
        for char in line.rstrip("\n"): # removing trailing newline
            characters.add(char)

# inspecting the character set
print(len(characters)) # 9442 
print(" " in characters)
print("\n" in characters)
print(sorted(characters)[:20])

# adding unknown character token <unk>
sorted_characters = sorted(characters) # ordered list
vocab = ["<unk>"] + sorted_characters
print(vocab[0])
print(len(vocab)) # 9443


But French people use the word county (comté) in the name of the Free County region, the old Free County of Burgundy.

<class 'str'>
118
9442
True
False
[' ', '!', '"', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4']
<unk>
9443


In [20]:
## creating the mappings

#for i, char in enumerate(vocab):
    #print(i, repr(char))

char_to_id = {char: i for i, char in enumerate(vocab)}
id_to_char = {i: char for i, char in enumerate(vocab)}

# testing mappings
print(char_to_id["<unk>"])
print(char_to_id["a"])
print(id_to_char[0])
print(id_to_char[char_to_id["a"]])

0
58
<unk>
a


In [21]:
## encoder

def encode(text):
    token_ids = []

    for char in text:
        if char in char_to_id:
            token_id = char_to_id[char]
        else:
            token_id = char_to_id.get("<unk>", 0) # fallback for <unk>
        token_ids.append(token_id)

    return token_ids


## decoder

def decode(encoded_text):
    chars = []

    for token_id in encoded_text:
        if token_id in id_to_char:
            char = id_to_char[token_id]
        else:
            char = id_to_char.get(0, "<unk>")
        chars.append(char)

    text = "".join(chars)
    return text


### BPE Tokenizers

In [22]:
## tokenizer imports

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [23]:
import sentencepiece as spm

TOKENIZER_DIR = Path("../artifacts/tokenizers")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

### Smaller BPE Tokenizer

Vocabulary size:  
Because:

In [ ]:
# configuring trainer
spm.SentencePieceTrainer.train(
    input = str(balanced_path),
    model_prefix = str(TOKENIZER_DIR / "small_bpe"),
    model_type = "bpe",
    vocab_size = 3000,
    character_coverage = 0.9929, # had to lower
    unk_id = 0,
    bos_id = -1,    # disable
    eos_id = -1,    # disable
)

# processor (object used for encoding)
small_bpe = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "small_bpe.model")
)

I0000 00:00:1790781752.073366 3860554 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /srv/data/lt2326-h26/a1/tokenizer/balanced.txt
  input_format: 
  model_prefix: ../artifacts/tokenizers/small_bpe
  model_type: BPE
  vocab_size: 3000
  self_test_sample_size: 0
  character_coverage: 0.9929
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: -1
  eos_id: -1
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piec

Vocabulary size: 3000

--- English ---
Text:   This is a simple tokenization example.
Pieces: ['▁T', 'h', 'is', '▁is', '▁a', '▁s', 'im', 'p', 'le', '▁to', 'k', 'en', 'i', 'z', 'ation', '▁', 'e', 'x', 'am', 'p', 'le', '.']
IDs:    [42, 130, 17, 76, 11, 10, 71, 141, 49, 60, 133, 9, 121, 151, 68, 118, 119, 257, 43, 141, 49, 138]

--- Turkish ---
Text:   Bu Türkçe bir örnek cümledir.
Pieces: ['▁B', 'u', '▁T', 'ü', 'r', 'k', 'ç', 'e', '▁bir', '▁', 'ö', 'r', 'n', 'ek', '▁c', 'ü', 'm', 'l', 'ed', 'ir', '.']
IDs:    [57, 132, 42, 147, 123, 133, 153, 119, 74, 118, 165, 123, 122, 56, 23, 147, 131, 125, 20, 14, 138]

--- Chinese ---
Text:   这是一个中文例句。
Pieces: ['▁', '这', '是', '一', '个', '中', '文', '例', '句', '。']
IDs:    [118, 318, 170, 159, 268, 168, 265, 602, 1801, 143]


I0000 00:00:1790781753.783924 3860554 trainer_interface.cc:702] Saving model: ../artifacts/tokenizers/small_bpe.model
I0000 00:00:1790781753.791964 3860554 trainer_interface.cc:714] Saving vocabs: ../artifacts/tokenizers/small_bpe.vocab


In [36]:
'''
## HuggingFace

# creating BPE model
small_bpe_model = BPE(unk_token="<unk>")

# creating BPE tokenizer
small_bpe = Tokenizer(small_bpe_model)

# choosing pre-tokenizer
small_bpe.pre_tokenizer = Whitespace()

#print(small_bpe.pre_tokenizer.pre_tokenize_str(
    #"Don't tokenize-this! 我喜欢机器学习。"
#))

# configuring trainer
small_bpe_trainer = BpeTrainer(
    special_tokens=["<unk>"],
    limit_alphabet = 1000,   # 1000 base characters -> + 1000 learned BPE merges
    vocab_size=2000,    # ?????
)

# training
training_data = [str(balanced_path)]
small_bpe.train(training_data, small_bpe_trainer)
'''

'\n## HuggingFace\n\n# creating BPE model\nsmall_bpe_model = BPE(unk_token="<unk>")\n\n# creating BPE tokenizer\nsmall_bpe = Tokenizer(small_bpe_model)\n\n# choosing pre-tokenizer\nsmall_bpe.pre_tokenizer = Whitespace()\n\n#print(small_bpe.pre_tokenizer.pre_tokenize_str(\n    #"Don\'t tokenize-this! 我喜欢机器学习。"\n#))\n\n# configuring trainer\nsmall_bpe_trainer = BpeTrainer(\n    special_tokens=["<unk>"],\n    limit_alphabet = 1000,   # 1000 base characters -> + 1000 learned BPE merges\n    vocab_size=2000,    # ?????\n)\n\n# training\ntraining_data = [str(balanced_path)]\nsmall_bpe.train(training_data, small_bpe_trainer)\n'

### Larger BPE Tokenizer


Vocabulary size:  
Because:

In [37]:
'''

## HuggingFace

# creating BPE model
big_bpe_model = BPE(unk_token="<unk>")

# creating BPE tokenizer
big_bpe = Tokenizer(big_bpe_model)

# choosing pre-tokenizer
big_bpe.pre_tokenizer = Whitespace()

print(big_bpe.pre_tokenizer.pre_tokenize_str(
    "Don't tokenize-this! 我喜欢机器学习。"
))

# configuring trainer
big_bpe_trainer = BpeTrainer(
    special_tokens=["<unk>"],
    limit_alphabet = 1000,   # 1000 base characters -> + 1000 learned BPE merges
    vocab_size=10000,    # ?????
)

# training
#training_data = [str(balanced_path)]
big_bpe.train(training_data, big_bpe_trainer)
'''

'\n\n## HuggingFace\n\n# creating BPE model\nbig_bpe_model = BPE(unk_token="<unk>")\n\n# creating BPE tokenizer\nbig_bpe = Tokenizer(big_bpe_model)\n\n# choosing pre-tokenizer\nbig_bpe.pre_tokenizer = Whitespace()\n\nprint(big_bpe.pre_tokenizer.pre_tokenize_str(\n    "Don\'t tokenize-this! 我喜欢机器学习。"\n))\n\n# configuring trainer\nbig_bpe_trainer = BpeTrainer(\n    special_tokens=["<unk>"],\n    limit_alphabet = 1000,   # 1000 base characters -> + 1000 learned BPE merges\n    vocab_size=10000,    # ?????\n)\n\n# training\n#training_data = [str(balanced_path)]\nbig_bpe.train(training_data, big_bpe_trainer)\n'

In [ ]:
# configuring the trainer
spm.SentencePieceTrainer.train(
    input = str(balanced_path),
    model_prefix = str(TOKENIZER_DIR / "large_bpe"),
    model_type = "bpe",
    vocab_size = 10000,
    character_coverage = 0.9929, # recommended for Chinese
    unk_id = 0,
    bos_id = -1,    # disable
    eos_id = -1,    # disable
)

# processor (object used for encoding)
large_bpe = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "large_bpe.model")
)

I0000 00:00:1790781776.044459 3860554 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /srv/data/lt2326-h26/a1/tokenizer/balanced.txt
  input_format: 
  model_prefix: ../artifacts/tokenizers/large_bpe
  model_type: BPE
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9929
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: -1
  eos_id: -1
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_pie

Vocabulary size 10000


I0000 00:00:1790781778.511093 3860554 trainer_interface.cc:702] Saving model: ../artifacts/tokenizers/large_bpe.model
I0000 00:00:1790781778.519770 3860554 trainer_interface.cc:714] Saving vocabs: ../artifacts/tokenizers/large_bpe.vocab


In [41]:
## sanity checking both BPE tokenizers

# checking vocab sizes
print("Small BPE vocab:", small_bpe.vocab_size())
print("Large BPE vocab:", large_bpe.vocab_size())
print("Small UNK ID:", small_bpe.unk_id())
print("Large UNK ID:", large_bpe.unk_id())

# looking at tokenization examples
examples = {
    "English": "This is a simple tokenization example.",
    "Turkish": "Bu Türkçe bir örnek cümledir.",
    "Chinese": "这是一个中文例句。"
}

for language, sentence in examples.items():
    print(f"\n=== {language} ===")
    
    print("SMALL:")
    print(small_bpe.encode(sentence, out_type=str))
    
    print("LARGE:")
    print(large_bpe.encode(sentence, out_type=str))

Small BPE vocab: 3000
Large BPE vocab: 10000
Small UNK ID: 0
Large UNK ID: 0

=== English ===
SMALL:
['▁T', 'h', 'is', '▁is', '▁a', '▁s', 'im', 'p', 'le', '▁to', 'k', 'en', 'i', 'z', 'ation', '▁', 'e', 'x', 'am', 'p', 'le', '.']
LARGE:
['▁This', '▁is', '▁a', '▁sim', 'ple', '▁to', 'ken', 'ization', '▁example', '.']

=== Turkish ===
SMALL:
['▁B', 'u', '▁T', 'ü', 'r', 'k', 'ç', 'e', '▁bir', '▁', 'ö', 'r', 'n', 'ek', '▁c', 'ü', 'm', 'l', 'ed', 'ir', '.']
LARGE:
['▁Bu', '▁Türk', 'çe', '▁bir', '▁örnek', '▁c', 'üm', 'led', 'ir', '.']

=== Chinese ===
SMALL:
['▁', '这', '是', '一', '个', '中', '文', '例', '句', '。']
LARGE:
['▁这', '是一个', '中文', '例', '句', '。']


### Evaluating the tokenizers on the validation data

In [ ]:
## encoding the data with all three tokenizers

def encode_char(text):
    return encode(text)

def encode_small_bpe(text):
    return small_bpe.encode(text, out_type=int)

def encode_large_bpe(text):
    return large_bpe.encode(text, out_type=int)

In [ ]:
## evaluation function

def evaluate_tokenizer(encode_fn, validation_path, unk_id):